# Forward temperature synthesis: a visual companion

This notebook explains `test_02_forward.py`. T255 temperature coefficients are truncated and synthesized onto the F32/T63 or F96/T191 full Gaussian grid, using MIR-compatible synthesis. Temperature is the only variable asserted by this file; both levels, 96 and 137, are tested.

**Pass criterion:** `xr.testing.assert_allclose(computed, reference, rtol=1e-5, atol=0)`. The maps and error spectra explain the result; they do not add spectral pass criteria or test the vector transformations.

Use the repository's Pixi Python environment, or the Python dev environment with Matplotlib and Cartopy installed. Run all cells in order with the kernel's working directory set to `tests`, so the neighboring helper modules can be imported directly. Coastline rendering may require downloaded map data.

In [ ]:
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

plt.rcParams.update({'figure.dpi': 110})

## 1. Choose a case

Change the grid or level and rerun the following cells. F32 has 64 latitudes and 128 longitudes (T63); F96 has 192 latitudes and 384 longitudes (T191). The final section checks all four grid/level combinations.

The default subplot projection is `ccrs.PlateCarree()`. Other Cartopy projections can be used without changing the longitude/latitude data coordinates. Set `projection = None` for plain Matplotlib axes: no mesh transform or coastlines are then used, regardless of the coastline toggle.

Coastlines are optional and may require Cartopy's downloaded map data.

In [ ]:
import common
from plotting import make_ps_plot, show_map
from test_02_forward import construct_transformation, test_forward_mir

grid = 'f32'
level = 96
projection = ccrs.PlateCarree()
coastlines = True
rtol = 1e-5
atol = 0.0

if grid not in {'f32', 'f96'} or level not in {96, 137}:
    message = 'Choose f32/f96 and level 96/137'
    raise ValueError(message)

## 2. Load the spectral input and grid reference

The readers and transform constructor are shared with pytest. Reference latitude labels can differ from computed nodes by rounding bits: verify agreement to 1e-12 degrees, then use canonical coordinates before subtraction so xarray cannot silently intersect different labels. The assertion later also runs against the original reference.

In [ ]:
ds_spec = common.open_decoded('t255')
ds_grid_ref_raw = common.open_decoded(grid)
transformation = construct_transformation(ds_grid_ref_raw, ds_spec)
np.testing.assert_allclose(
    ds_grid_ref_raw.latitude,
    transformation.grid.lat,
    rtol=0,
    atol=1e-12,
)
np.testing.assert_allclose(
    ds_grid_ref_raw.longitude,
    transformation.grid.lon,
    rtol=0,
    atol=1e-12,
)
ds_grid_ref = (
    ds_grid_ref_raw[['t']]
    .assign_coords(
        latitude=transformation.grid.lat,
        longitude=transformation.grid.lon,
    )
    .compute()
)

pd.DataFrame([
    {
        'dataset': 'Spectral source',
        'truncation': int(ds_spec.truncation),
        'dimensions': dict(ds_spec.sizes),
        'levels': ds_spec.level.to_numpy().tolist(),
    },
    {
        'dataset': grid.upper(),
        'truncation': int(ds_grid_ref_raw.truncation),
        'dimensions': dict(ds_grid_ref_raw.sizes),
        'levels': ds_grid_ref.level.to_numpy().tolist(),
    },
]).set_index('dataset')

## 3. Apply the transformation in two visible steps

`fold_clm` converts packed coefficients into real/imaginary components with degree and order axes, retaining degrees up to the grid truncation. MIR synthesis then omits the coefficient pair at the highest sectoral mode: degree = order = truncation.

The two-step result must agree with the public method called by the test.

In [ ]:
ds_folded = transformation.fold_clm(ds_spec[['t']]).compute()
ds_grid_steps = transformation.folded_spec_to_grid_mir(ds_folded).compute()
ds_grid_mir = transformation.unfolded_spec_to_grid_mir(ds_spec[['t']]).compute()
xr.testing.assert_allclose(ds_grid_steps, ds_grid_mir, rtol=0, atol=0)
ds_grid_mir

## 4. Inspect spatial errors

Reference and computed temperature share a color scale. The signed residual has its own symmetric scale, otherwise the small error would be invisible beside the temperature field. Residual means computed minus reference.

In [ ]:
figure_map = show_map(
    ds_grid_ref,
    ds_grid_mir,
    't',
    level,
    projection=projection,
    coastlines=coastlines,
)

## 5. What does the assertion allow?

At each point the test requires $|t_{new}-t_{ref}| \leq 10^{-5}|t_{ref}|$. Define $q=|t_{new}-t_{ref}|/(10^{-5}|t_{ref}|)$: points with $q\leq1$ pass. With a zero reference, zero residual gives q = 0 and a nonzero residual gives infinity.

The table reports unweighted RMS, Gaussian-weighted RMS, and conventional relative RMS (error RMS divided by reference RMS). These are diagnostics, not the pass criterion. The legacy helper's symmetric relative-RMS number uses a different denominator.

In [ ]:
def tolerance_ratio(actual, reference):
    actual, reference = xr.align(actual, reference, join='exact')
    residual = abs(actual - reference)
    limit = atol + rtol * abs(reference)
    return xr.where(
        limit > 0,
        residual / limit.where(limit > 0, 1),
        xr.where(residual == 0, 0, np.inf),
    )


def error_metrics(actual, reference, weights):
    actual, reference = xr.align(actual, reference, join='exact')
    residual = actual - reference
    ratio = tolerance_ratio(actual, reference)
    gauss = xr.DataArray(
        weights,
        dims='latitude',
        coords={'latitude': reference.latitude},
    )
    rms = float(np.sqrt(np.square(residual).mean()))
    reference_rms = float(np.sqrt(np.square(reference).mean()))
    return {
        'max_abs_error': float(abs(residual).max()),
        'rms_error': rms,
        'weighted_rms_error': float(
            np.sqrt(np.square(residual).weighted(gauss).mean()),
        ),
        'relative_rms': rms / reference_rms
        if reference_rms > 0
        else (0 if rms == 0 else np.inf),
        'max_tolerance_ratio': float(ratio.max()),
        'fraction_failing': float((ratio > 1).mean()),
    }


selected_new = ds_grid_mir.t.sel(level=level)
selected_ref = ds_grid_ref.t.sel(level=level)
ratio = tolerance_ratio(selected_new, selected_ref)
xr.testing.assert_allclose(
    ds_grid_mir[['t']],
    ds_grid_ref_raw[['t']].compute(),
    rtol=rtol,
    atol=atol,
)
pd.DataFrame(
    [error_metrics(selected_new, selected_ref, transformation.grid.gauss_weights)],
    index=[f'{grid}, level {level}'],
)

In [ ]:
figure_ratio, axis_ratio = plt.subplots(figsize=(9, 3.5), layout='constrained')
mesh_ratio = axis_ratio.pcolormesh(
    ratio.longitude,
    ratio.latitude,
    ratio,
    shading='auto',
    vmin=0,
    vmax=max(1.0, float(ratio.max())),
    cmap='magma',
)
figure_ratio.colorbar(
    mesh_ratio,
    ax=axis_ratio,
    label='Pointwise tolerance ratio q (pass <= 1)',
    orientation='horizontal',
    location='bottom',
    pad=0.15,
)
axis_ratio.set(
    title=f'{grid.upper()}, level {level}: actual test criterion',
    xlabel='Longitude (degrees)',
    ylabel='Latitude (degrees)',
)
plt.show()

## 6. Error spectra: explanatory, not an extra assertion

Analyze the reference, computed temperature, and their residual using the scalar transform. `common.degree_power` sums squared coefficients with zonal weight 1 and nonzonal weight 2. The residual spectrum is not the difference of the two field spectra: equal power can conceal phase errors.

The left plot includes degree zero, which dominates temperature power. The right plot shows relative residual amplitude per degree, revealing small-scale errors hidden by the mean. Zero power is omitted on logarithmic y axes; zero reference power makes the relative diagnostic undefined. Scalar analysis only resolves retained degrees, so spatial errors remain important. No spectral tolerance is imposed by this test.

In [ ]:
ds_residual = ds_grid_mir - ds_grid_ref
power_ref = common.degree_power(
    transformation.grid_to_folded_spec(ds_grid_ref),
).compute()
power_new = common.degree_power(
    transformation.grid_to_folded_spec(ds_grid_mir),
).compute()
power_error = common.degree_power(
    transformation.grid_to_folded_spec(ds_residual),
).compute()
figure_spectrum = make_ps_plot(power_ref, power_new, power_error, 't', level)
plt.show()

## 7. Why the MIR variant?

This is an explanatory extension, not another assertion in `test_02_forward.py`. Ordinary synthesis retains the highest sectoral mode. MIR synthesis suppresses it to match the reference convention.

Isolate that mode and verify that its synthesized contribution equals ordinary minus MIR synthesis. Reference residuals may have other causes: not every discrepancy is explained by this omission.

In [ ]:
ds_grid_ordinary = transformation.folded_spec_to_grid(ds_folded).compute()
ds_highest = ds_folded.copy(deep=True)
highest_coefficients = ds_highest.t.data[..., :, -1, -1].copy()
ds_highest.t.data[:] = 0
ds_highest.t.data[..., :, -1, -1] = highest_coefficients
ds_highest_grid = transformation.folded_spec_to_grid(ds_highest).compute()
ds_mir_difference = ds_grid_ordinary - ds_grid_mir
xr.testing.assert_allclose(ds_mir_difference, ds_highest_grid, rtol=1e-8, atol=1e-12)

figure_mir = show_map(
    ds_grid_mir,
    ds_grid_ordinary,
    't',
    level,
    projection=projection,
    coastlines=coastlines,
    title=f'Ordinary versus MIR synthesis, {grid.upper()}, level {level}',
    panel_titles=('MIR synthesis', 'Ordinary synthesis', 'Highest-mode contribution'),
)

pd.DataFrame([
    {
        'method': 'MIR (tested)',
        **error_metrics(selected_new, selected_ref, transformation.grid.gauss_weights),
    },
    {
        'method': 'Ordinary (diagnostic)',
        **error_metrics(
            ds_grid_ordinary.t.sel(level=level),
            selected_ref,
            transformation.grid.gauss_weights,
        ),
    },
]).set_index('method')

## 8. Run the real test and summarize every case

Call `test_forward_mir` itself for both grids, including both levels in each call exactly as pytest does. A failed assertion stops execution rather than being hidden in a pass table. Then report each level separately.

To inspect another case visually, change the parameters in section 1 and rerun. The pass/fail statement concerns pointwise temperature agreement with these references, not vector-transform correctness or guaranteed accuracy for arbitrary inputs.

In [ ]:
rows = []
for grid_name in ('f32', 'f96'):
    grid_reference_raw = common.open_decoded(grid_name)
    transform_case = construct_transformation(grid_reference_raw, ds_spec)
    test_forward_mir(ds_spec, grid_reference_raw)
    np.testing.assert_allclose(
        grid_reference_raw.latitude,
        transform_case.grid.lat,
        rtol=0,
        atol=1e-12,
    )
    np.testing.assert_allclose(
        grid_reference_raw.longitude,
        transform_case.grid.lon,
        rtol=0,
        atol=1e-12,
    )
    grid_reference = (
        grid_reference_raw[['t']]
        .assign_coords(
            latitude=transform_case.grid.lat,
            longitude=transform_case.grid.lon,
        )
        .compute()
    )
    grid_result = transform_case.unfolded_spec_to_grid_mir(ds_spec[['t']]).compute()
    rows.extend([
        {
            'grid': grid_name,
            'level': case_level,
            'status': 'PASS',
            **error_metrics(
                grid_result.t.sel(level=case_level),
                grid_reference.t.sel(level=case_level),
                transform_case.grid.gauss_weights,
            ),
        }
        for case_level in (96, 137)
    ])
summary = pd.DataFrame(rows).set_index(['grid', 'level'])
summary